In [1]:
import math

In [2]:
class Value:
    def __init__(self, data, _children=(), _op='', label='') -> None: 
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self._label = label

    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(
            self.data + other.data, 
            _children=(self, other),
            _op='+'
        )

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward 
        
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(
            data=self.data * other.data,
            _children=(self, other),
            _op=('*')
        )

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out

    def __rmul__(self, other):
        return self * other

    def exp(self):
        x = self.data
        out = Value(
            data=math.e ** self.data,
            _children=(self, ),
            _op='exp'
        )

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        
        return out 

    def __pow__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        out = Value(
            data=self.data ** other.data,
            _children=(self, other),
            _op=('^')
        )
        def _backward():
            self.grad += other.data * self.data ** (other.data - 1) * out.grad
            other.grad += math.log(self.data) * out.data * out.grad
        out._backward = _backward

        return out

    def __rpow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other ** self

    def __truediv__(self, other):
        return self * other ** -1

    def __neg__(self):
        return self * -1

    def __sub__ (self, other):
        return self + (-other)

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(
            data=t,
            _children=(self, ),
            _op='tanh',
        )

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        
        return out
            
        

In [3]:
"""inputs x_1, x_2"""
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

"""weights w_1, w_2"""
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

"""bias"""
b = Value(6.8813735870195432, label='b')

"""calculation"""
x1w1 = x1 * w1; x1w1.label = 'x1w1'
x2w2 = x2 * w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1x2w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [4]:
o.grad = 1.0
o._backward()

In [5]:
n.grad

0.4999999999999999